In [1]:
import pandas as pd, sqlite3

con = sqlite3.connect("../data/clean/posts.db")
raw = pd.read_csv("../data/raw/USvideos.csv")

sample = pd.read_sql("""
SELECT video_id, title, views, likes, comment_count, engagement_rate AS sql_er
FROM v_valid_posts
""", con).sample(10, random_state=7)

sample

,video_id,title,views,likes,comment_count,sql_er
1648,uTH--XRCGxM,5 Strange Things That Happened to Stranger Thi...,127404,3869,111,3.1239
623,4CGBSGEkbKA,Oprah Winfrey - 2018 Golden Globes - Full Back...,1144298,6809,2350,0.8004
1317,dRpNZV18N_g,SPAGHETTI BURRITO VS SPAGHETTI BURRITO,324024,10288,1586,3.6645
971,QRq0pkr2TWM,Cheap Thrills - Sia / Tina Boo Choreography,627933,28580,444,4.6221
1698,ZEWi-0Q_bpg,Cavs players reportedly upset with LeBron; did...,107113,965,575,1.4377
1895,j5cSKF5FzD0,Bradley Beal Goes OFF For a Career-High 51 Poi...,41159,658,169,2.0093
1714,qmYcbgJBv7s,Flower (2018) | Official US Trailer HD,100812,791,66,0.8501
1559,2X5fG1A4kuc,Nick Foles used basketball to lead Super Bowl-...,172027,1548,294,1.0708
1831,lyavSkF30Rs,Ashanti Speaks on Irv Gotti and Murder Inc | E...,57465,456,176,1.0998
1137,kOsl3cmK3zg,Google Search Team Answers the Web's Most Sear...,453506,14494,916,3.3980


In [2]:
rows = []
for _, r in sample.iterrows():
    # take the raw row with the highest views (same rule used in cleaning)
    raw_row = raw[raw["video_id"] == r["video_id"]].sort_values("views", ascending=False).iloc[0]
    manual = (raw_row["likes"] + raw_row["comment_count"]) / raw_row["views"] * 100
    rows.append({
        "video_id": r["video_id"],
        "raw_views": raw_row["views"],
        "raw_likes": raw_row["likes"],
        "raw_comments": raw_row["comment_count"],
        "manual_er": round(manual, 4),
        "sql_er": r["sql_er"],
        "match": abs(round(manual, 4) - r["sql_er"]) < 0.0001,
    })

check = pd.DataFrame(rows)
check

,video_id,raw_views,raw_likes,raw_comments,manual_er,sql_er,match
0,uTH--XRCGxM,127404,3869,111,3.1239,3.1239,True
1,4CGBSGEkbKA,1144298,6809,2350,0.8004,0.8004,True
2,dRpNZV18N_g,324024,10288,1586,3.6645,3.6645,True
3,QRq0pkr2TWM,627933,28580,444,4.6221,4.6221,True
4,ZEWi-0Q_bpg,107113,965,575,1.4377,1.4377,True
5,j5cSKF5FzD0,41159,658,169,2.0093,2.0093,True
6,qmYcbgJBv7s,100812,791,66,0.8501,0.8501,True
7,2X5fG1A4kuc,172027,1548,294,1.0708,1.0708,True
8,lyavSkF30Rs,57465,456,176,1.0998,1.0998,True
9,kOsl3cmK3zg,453506,14494,916,3.3980,3.3980,True


In [3]:
check.to_csv("../docs/verification_table.csv", index=False)
print("All 10 match:", check["match"].all())

All 10 match: True


In [4]:
pd.read_sql("""
SELECT
  SUM(is_zero_views) AS zero_view_videos,
  SUM(is_deleted)    AS deleted_videos,
  SUM(is_outlier)    AS viral_outliers,
  COUNT(*)           AS total_videos
FROM posts
""", con)

,zero_view_videos,deleted_videos,viral_outliers,total_videos
0,0,2,225,2073
